# Deep Learning Master Class — Colab Uygulaması
## Yüz analizi ve nesne tanıma: bir fotoğraftan kameraya

**Kuantum Bootcamp · Veysel Murat Görken**

---

Bugün konuştuğumuz her şey burada bir araya geliyor:

| Sunumda gördüğümüz | Bu defterde |
|---|---|
| CNN — filtreler görüntüde gezer | RetinaFace de, YOLO da bir CNN |
| Ölçekleme / normalizasyon | Görüntü 224x224'e getirilip 0–1 aralığına çekiliyor |
| Softmax — olasılık dağılımı | Duygu tahmini 7 sınıfa dağılmış olasılık |
| Regresyon vs sınıflandırma | Yaş tek bir sayı, cinsiyet iki sınıflı olasılık |
| Embedding — anlam bir vektöre dönüşür | Yüz de 512 boyutlu bir vektöre dönüşüyor |
| Üretken vs ayırt edici modeller | Üretilmiş bir yüzü ayırt edici bir modele veriyoruz |
| AlexNet → ImageNet → bugün | YOLO: tek bakışta 80 nesne türü, kamerada canlı |

**Hiçbir şey eğitmiyoruz.** Eğitilmiş modelleri kullanıyoruz — bugün sahadaki
işlerin çoğu böyle başlıyor.

> Bu defter, [generative-ai-workshop](https://github.com/grknc/generative-ai-workshop)
> projesindeki **Yüz Analizi** sekmesinin Python karşılığıdır. Orada modeller
> tarayıcıda (face-api.js) çalışıyor; burada Colab'da `DeepFace` ve `YOLO` ile.

**Akış:** 0 Kurulum → 1–4 Yüz analizi → 5 Kendi fotoğrafınız → 6 Yüz vektörü →
7 YOLO ile nesne tespiti → 8 Kamera → 9 Canlı nesne takibi → 10 Sınırlar

---
## 0 · Kurulum

Colab'da hazır gelen hiçbir pakete **dokunmuyoruz**: TensorFlow, PyTorch, OpenCV,
NumPy olduğu gibi kalıyor. Yalnızca Colab'da olmayan 5 küçük paketi, sabit
sürümleriyle ve **bağımlılıksız** (`--no-deps`) ekliyoruz. Kurulum ~5 saniye,
oturum yeniden başlatılmıyor.

| Paket | Ne için |
|---|---|
| `deepface` | yüz analizi: duygu, yaş, cinsiyet, yüz vektörü |
| `retina-face` | CNN tabanlı yüz bulucu |
| `ultralytics` (+ `ultralytics-thop`) | YOLO nesne tespiti ve takibi |
| `lap` | takipte kutuları kareler arasında eşleştirme |

> **Neden `--no-deps`?** Normal kurulumda bu paketler kendi OpenCV / ONNX
> sürümlerini çekip Colab'ın kurulu paketlerini değiştiriyor — eski sürümde
> yaşanan `CascadeClassifier` hatası buradan geliyordu. İhtiyaç duydukları her
> şey Colab'da zaten var. (`pip`'in "requires ... not installed" uyarıları
> bilinçli; o paketleri kullanmıyoruz.)

In [ ]:
import subprocess, sys

PAKETLER = ["deepface==0.0.103", "retina-face==0.0.19",
            "ultralytics==8.4.190", "ultralytics-thop==2.2.3", "lap==0.5.13"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *PAKETLER], check=True)
print("✓ Kurulum tamam — Colab'ın kendi paketlerine dokunulmadı.")

### Doğrulama

Kullandığımız Colab paketlerinin sürümlerini yazdırıp iki kütüphanenin de
sorunsuz yüklendiğini kontrol edelim.

In [ ]:
import sys, importlib.metadata as md

print("Python      :", sys.version.split()[0])
for paket in ["deepface", "retina-face", "ultralytics", "lap",
              "tensorflow", "tf_keras", "torch", "numpy", "opencv-python"]:
    try:
        print(f"{paket:<12}:", md.version(paket))
    except md.PackageNotFoundError:
        print(f"{paket:<12}: YOK  ← kurulum hücresini tekrar çalıştırın")

### Kütüphaneler ve yardımcılar

Bütün importlar, Türkçe etiketler ve renk paleti burada.

In [ ]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["DEEPFACE_LOG_LEVEL"] = "40"   # yalnızca hataları göster
os.environ["YOLO_VERBOSE"] = "False"
os.environ["YOLO_AUTOINSTALL"] = "False"  # YOLO ders sırasında kendi kendine paket kurmasın

import time, json, requests, numpy as np
import importlib.metadata as md
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.font_manager as fm
from io import BytesIO
from base64 import b64decode, b64encode
from collections import Counter
from PIL import Image, ImageDraw, ImageFont, ImageEnhance

# Sıra önemli: PyTorch tarafı (YOLO) TensorFlow'dan (DeepFace) ÖNCE yüklenmeli.
# GPU çalışma zamanında ters sıra, triton kütüphanesiyle çakışıp oturumu çökertebiliyor.
import torch, torchvision.ops
try:
    import triton            # yalnızca GPU çalışma zamanında var
except ImportError:
    pass
from ultralytics import YOLO
from ultralytics.utils import ASSETS
from deepface import DeepFace

ALTIN, LACIVERT, TURKUAZ = "#C08A2E", "#15263F", "#148F77"
RENKLER = [ALTIN, TURKUAZ, LACIVERT, "#B03A2E", "#6C3483", "#1F618D", "#7D6608", "#117864"]
YAZI_DOSYASI = fm.findfont("DejaVu Sans")

DUYGU_TR = {"angry": "kızgın", "disgust": "tiksinti", "fear": "korku", "happy": "mutlu",
            "sad": "üzgün", "surprise": "şaşkın", "neutral": "nötr"}
CINSIYET_TR = {"Man": "Erkek", "Woman": "Kadın"}

print("Kütüphaneler hazır.")

### Modelleri önceden indir

Bütün model ağırlıkları (~1,3 GB) burada bir kez indirilir: yüz bulucu, duygu,
yaş, cinsiyet, yüz vektörü ve YOLO. Colab'da ~1 dakika sürer.

> **İpucu:** Dersten birkaç dakika önce bu hücreye kadar çalıştırın; ders
> sırasında hiçbir hücre indirme beklemez.

In [ ]:
bas = time.time()
DETEKTOR = "retinaface"          # CNN tabanlı yüz bulucu
ornek_yuz = str(ASSETS / "zidane.jpg")   # ultralytics ile gelen örnek fotoğraf
DeepFace.analyze(ornek_yuz, actions=["emotion", "age", "gender"],
                 detector_backend=DETEKTOR, enforce_detection=False, silent=True)
DeepFace.represent(ornek_yuz, model_name="Facenet512", detector_backend=DETEKTOR, enforce_detection=False)
YOLO_AGIRLIK = "yolo26n.pt"
yolo = YOLO(YOLO_AGIRLIK)
yolo(ASSETS / "bus.jpg", verbose=False)
print(f"✓ Bütün modeller hazır ({time.time() - bas:.0f} sn).")

---
## 1 · Bir yüz bulalım

Demo için **yapay üretilmiş** bir yüz kullanıyoruz. Aşağıdaki fotoğraftaki kişi
gerçekte yok — bir GAN tarafından üretildi. Her çalıştırmada farklı bir yüz gelir.

> **Konuşma notu:** Sunumdaki 'üretken vs klasik' slaydını hatırlayın. Bu fotoğrafı
> bir **üretken** model üretti; şimdi onu bir **ayırt edici** modele vereceğiz.
> İki dünya aynı hücrede karşılaşıyor.

Bağlantı çalışmazsa hücre sizden bilgisayarınızdan bir fotoğraf seçmenizi ister;
akış onunla devam eder.

In [ ]:
URET_URL = "https://thispersondoesnotexist.com/random-person.jpeg"


def yuz_indir(url=URET_URL):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (DL MasterClass)"}, timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


def resim_yukle():
    """Bilgisayardan bir veya birkaç fotoğraf seçtirir, PIL görüntüleri döndürür."""
    from google.colab import files
    yuklenen = files.upload()
    return [Image.open(BytesIO(icerik)).convert("RGB") for icerik in yuklenen.values()]


try:
    resim = yuz_indir()
except Exception as e:
    print("Üretilmiş yüz indirilemedi:", e, "\n→ Bilgisayarınızdan bir yüz fotoğrafı seçin:")
    resim = resim_yukle()[0]

resim.save("foto.jpg")
plt.figure(figsize=(5, 5))
plt.imshow(resim); plt.axis("off")
plt.title(f"Girdi — {resim.size[0]}x{resim.size[1]} piksel")
plt.show()

---
## 2 · Adım 1 — Yüzü bul (face detection)

Analiz yapmadan önce fotoğrafta yüzün **nerede** olduğunu bulmak gerekiyor.
Bunu **RetinaFace** yapıyor — bir **CNN**: küçük filtreler görüntünün üzerinde
geziyor ve "burada yüz var" diyor. Kimse ona "göz burunun üstündedir" diye bir
kural yazmadı; bunu binlerce etiketli yüzden kendisi öğrendi.

Bulunan bölge kırpılıp 224x224'e ölçekleniyor ve değerler 0–1 aralığına
çekiliyor — sunumdaki ev fiyatı örneğinde yaptığımız ölçeklemenin aynısı.

In [ ]:
def yuzleri_bul(dosya):
    yuzler = DeepFace.extract_faces(img_path=dosya, detector_backend=DETEKTOR, enforce_detection=False)
    return [y for y in yuzler if y["confidence"] > 0]


bas = time.time()
yuzler = yuzleri_bul("foto.jpg")
print(f"Bulunan yüz sayısı: {len(yuzler)}  ({time.time() - bas:.1f} sn)")
if not yuzler:
    print("⚠ Yüz bulunamadı — bölüm 1'i tekrar çalıştırıp başka bir fotoğraf deneyin.")

fig, eksen = plt.subplots(1, 2, figsize=(11, 5))
eksen[0].imshow(resim)
for y in yuzler:
    a = y["facial_area"]
    eksen[0].add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"],
                                         fill=False, edgecolor=ALTIN, lw=3))
eksen[0].set_title("Tespit edilen yüz"); eksen[0].axis("off")

yuz = yuzler[0]["face"] if yuzler else np.asarray(resim) / 255
kirpik = np.asarray(Image.fromarray((yuz * 255).astype("uint8")).resize((224, 224))) / 255  # modelin beklediği boyut
eksen[1].imshow(kirpik)
eksen[1].set_title("Kırpılmış ve ölçeklenmiş — modele giren görüntü")
eksen[1].axis("off")
plt.tight_layout(); plt.show()

print("Modele giren dizinin boyutu :", kirpik.shape, f"→ {kirpik.size:,} sayı".replace(",", "."))
print(f"Değer aralığı               : {kirpik.min():.2f} ... {kirpik.max():.2f}")

> **Konuşma notu:** Sağdaki görüntü artık bir fotoğraf değil, `224 x 224 x 3` boyutunda
> bir sayı kümesi — yani 150.528 sayı. Ev fiyatı örneğinde 4 sayı vardı; fark sadece bu.
> Model fotoğraf görmez, sayı görür.

---
## 3 · Adım 2 — Duygu analizi (softmax çalışıyor)

Duygu modeli 7 sınıf üretiyor: kızgın, tiksinti, korku, mutlu, üzgün, şaşkın, nötr.
Çıkışta **softmax** var — tam olarak sunumda konuştuğumuz gibi, olasılıkların
toplamı 1 (yani %100) oluyor.

In [ ]:
def analiz(dosya, eylemler):
    return DeepFace.analyze(img_path=dosya, actions=eylemler, detector_backend=DETEKTOR,
                            enforce_detection=False, silent=True)[0]


def duygu_grafigi(duygular, eksen=None, baslik="Duygu dağılımı (softmax çıktısı)"):
    sirali = sorted(duygular.items(), key=lambda x: x[1])
    eksen = eksen or plt.figure(figsize=(7, 3.2)).gca()
    eksen.barh([DUYGU_TR.get(a, a) for a, _ in sirali], [p for _, p in sirali], color=TURKUAZ)
    eksen.set_xlabel("olasılık (%)"); eksen.set_xlim(0, 100); eksen.set_title(baslik)


duygu = analiz("foto.jpg", ["emotion"])

print("Baskın duygu:", DUYGU_TR.get(duygu["dominant_emotion"], duygu["dominant_emotion"]).upper())
print("-" * 34)
for ad, p in sorted(duygu["emotion"].items(), key=lambda x: -x[1]):
    print(f"  %{p:5.1f}   {DUYGU_TR.get(ad, ad)}")

duygu_grafigi(duygu["emotion"]); plt.tight_layout(); plt.show()
print(f"Olasılıkların toplamı: %{sum(duygu['emotion'].values()):.1f}  → softmax")

---
## 4 · Adım 3 — Yaş ve cinsiyet

Aynı yüz, iki farklı modele daha giriyor.

- **Yaş** — regresyon problemi: çıkış tek bir sayı (ev fiyatı örneğindeki gibi)
- **Cinsiyet** — ikili sınıflandırma: iki sınıfa dağıtılmış olasılık

In [ ]:
hepsi = analiz("foto.jpg", ["age", "gender", "emotion"])
g, d = hepsi["dominant_gender"], hepsi["dominant_emotion"]

print("=" * 40)
print(f"  Yaş tahmini    : {hepsi['age']}")
print(f"  Cinsiyet       : {CINSIYET_TR.get(g, g)}  (%{hepsi['gender'][g]:.1f})")
print(f"  Duygu          : {DUYGU_TR.get(d, d)}  (%{hepsi['emotion'][d]:.1f})")
print("=" * 40)

plt.figure(figsize=(4.5, 4.5))
plt.imshow(resim); plt.axis("off")
plt.title(f"{hepsi['age']} yaş · {CINSIYET_TR.get(g, g)} · {DUYGU_TR.get(d, d)}", fontsize=13)
plt.show()

---
## 5 · Kendi fotoğrafınızı deneyin (bilgisayardan yükleme)

Aşağıdaki hücreyi çalıştırıp **"Dosya seç"** düğmesiyle bilgisayarınızdan bir veya
birkaç fotoğraf seçin. 2–4. adımların hepsi tek seferde çalışır: fotoğraftaki
**bütün yüzler** bulunur, her birine yaş · cinsiyet · duygu yazılır, en büyük
yüzün duygu dağılımı yanında gösterilir.

Fotoğraf Colab oturumunuzda kalır, hiçbir yere gönderilmez; oturumu kapattığınızda silinir.

In [ ]:
def foto_analiz_et(resim, baslik=""):
    """Bir PIL görüntüsündeki bütün yüzleri bulur; yaş, cinsiyet, duyguyu tek figürde gösterir."""
    resim.save("_analiz.jpg")
    sonuclar = DeepFace.analyze("_analiz.jpg", actions=["age", "gender", "emotion"],
                                detector_backend=DETEKTOR, enforce_detection=False, silent=True)
    yuzler = [s for s in sonuclar if s.get("face_confidence", 0) > 0]
    if not yuzler:
        print("⚠ Bu fotoğrafta yüz bulunamadı — daha net, önden çekilmiş bir fotoğraf deneyin.")
        plt.figure(figsize=(5, 5)); plt.imshow(resim); plt.axis("off"); plt.show()
        return []

    fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.4, 1]})
    sol.imshow(resim); sol.axis("off")
    for s in yuzler:
        a, g, d = s["region"], s["dominant_gender"], s["dominant_emotion"]
        sol.add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"], fill=False, edgecolor=ALTIN, lw=3))
        sol.text(a["x"], a["y"] - 6, f"{s['age']} · {CINSIYET_TR.get(g, g)} · {DUYGU_TR.get(d, d)}",
                 color="white", fontsize=10, backgroundcolor=LACIVERT)
    sol.set_title(baslik or f"{len(yuzler)} yüz bulundu")

    en_buyuk = max(yuzler, key=lambda s: s["region"]["w"] * s["region"]["h"])
    duygu_grafigi(en_buyuk["emotion"], sag, "En büyük yüzün duygu dağılımı")
    plt.tight_layout(); plt.show()
    return yuzler


for kendi in resim_yukle():
    foto_analiz_et(kendi)

---
## 6 · Bonus — Yüz de bir vektör

Sunumda kelimelerin vektöre dönüştüğünü konuştuk: *kral − erkek + kadın ≈ kraliçe.*
Yüzler için de aynı şey geçerli.

`Facenet512` modeli bir yüzü **512 sayıdan oluşan bir vektöre** çeviriyor. Aynı
kişinin farklı fotoğrafları bu uzayda birbirine yakın, farklı kişiler uzak düşüyor.
Yüz tanıma sistemleri tam olarak böyle çalışır: karşılaştırma fotoğraflar arasında
değil, **vektörler arasında** yapılır.

In [ ]:
def vektore_cevir(dosya):
    return np.array(DeepFace.represent(img_path=dosya, model_name="Facenet512",
                                       detector_backend=DETEKTOR, enforce_detection=False)[0]["embedding"])


vektor = vektore_cevir("foto.jpg")
print("Vektör boyutu :", vektor.shape[0])
print("İlk 8 değer   :", "  ".join(f"{v:+.3f}" for v in vektor[:8]))

plt.figure(figsize=(11, 1.6))
plt.imshow(vektor.reshape(1, -1), aspect="auto", cmap="RdYlGn")
plt.yticks([]); plt.xlabel("512 boyut"); plt.title("Bu yüzün embedding vektörü")
plt.colorbar(); plt.tight_layout(); plt.show()

Şimdi iki karşılaştırma yapalım: aynı fotoğrafın **parlaklığı değiştirilmiş** hâli
ve **bambaşka** bir yüz. Ölçü **kosinüs benzerliği**: 1'e yakınsa vektörler aynı
yöne bakıyor, yani aynı kişi.

In [ ]:
ESIK = 0.70   # Facenet512 için kosinüs benzerliği eşiği (DeepFace'in mesafe eşiği 0.30 → 1 − 0.30)


def kosinus_benzerligi(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def karsilastir(vektor_a, dosya_b, baslik):
    benzerlik = kosinus_benzerligi(vektor_a, vektore_cevir(dosya_b))
    karar = "AYNI KİŞİ" if benzerlik >= ESIK else "FARKLI KİŞİ"
    print(f"{baslik:<30} benzerlik = {benzerlik:.3f}  (eşik {ESIK})  →  {karar}")
    return benzerlik


ImageEnhance.Brightness(resim).enhance(0.6).save("ayni_kisi.jpg")   # aynı kişi, farklı ışık
try:
    yuz_indir().save("baska_kisi.jpg")                               # başka bir üretilmiş yüz
except Exception:
    resim.transpose(Image.FLIP_LEFT_RIGHT).rotate(90).save("baska_kisi.jpg")
    print("(İkinci yüz indirilemedi; yerine döndürülmüş görüntü kullanılıyor.)")

b1 = karsilastir(vektor, "ayni_kisi.jpg", "Aynı yüz, farklı aydınlatma")   # vektor: yukarıdaki hücreden
b2 = karsilastir(vektor, "baska_kisi.jpg", "Farklı iki yüz")

fig, eksen = plt.subplots(1, 3, figsize=(10, 3.6))
for e, dosya, yazi in zip(eksen, ["foto.jpg", "ayni_kisi.jpg", "baska_kisi.jpg"],
                          ["Referans", f"benzerlik {b1:.2f}", f"benzerlik {b2:.2f}"]):
    e.imshow(Image.open(dosya)); e.set_title(yazi); e.axis("off")
plt.tight_layout(); plt.show()

> **Konuşma notu:** Aydınlatma değişti, pikseller tamamen farklı — ama vektörler
> hâlâ yakın. Model pikselleri değil, **kimliği** kodlamayı öğrenmiş. Telefonunuzun
> yüz tanıma özelliği de aynı prensiple çalışıyor.

---
## 7 · Yüzden nesneye — YOLO ile nesne tespiti

Şimdiye kadar tek bir şey aradık: yüz. **YOLO** (*You Only Look Once*) ise
görüntüye **tek bir bakışta** 80 farklı nesne türünü — insan, araba, köpek,
telefon, bardak... — konumlarıyla birlikte buluyor.

- Yine bir **CNN**: AlexNet'le başlayan çizginin bugünkü hâli
- Her kutu için iki çıktı var: **nerede** (koordinat — regresyon) ve **ne** (sınıf olasılığı)
- Kullandığımız `YOLO26n` modeli 2,4 milyon parametre, ~5 MB; CPU'da bile hızlı

> **Konuşma notu:** 2012'de AlexNet bir fotoğrafa *tek bir etiket* veriyordu.
> Bugün aynı fikir, saniyede onlarca kare üzerinde, her nesneyi ayrı ayrı bulup
> etiketliyor.

In [ ]:
# yolo = YOLO("yolo26n.pt") — "Modelleri önceden indir" hücresinde yüklendi

COCO_TR = {
    "person": "kişi", "bicycle": "bisiklet", "car": "araba", "motorcycle": "motosiklet",
    "airplane": "uçak", "bus": "otobüs", "train": "tren", "truck": "kamyon", "boat": "tekne",
    "traffic light": "trafik ışığı", "fire hydrant": "yangın musluğu", "stop sign": "dur tabelası",
    "parking meter": "parkmetre", "bench": "bank", "bird": "kuş", "cat": "kedi", "dog": "köpek",
    "horse": "at", "sheep": "koyun", "cow": "inek", "elephant": "fil", "bear": "ayı",
    "zebra": "zebra", "giraffe": "zürafa", "backpack": "sırt çantası", "umbrella": "şemsiye",
    "handbag": "el çantası", "tie": "kravat", "suitcase": "bavul", "frisbee": "frizbi",
    "skis": "kayak", "snowboard": "snowboard", "sports ball": "top", "kite": "uçurtma",
    "baseball bat": "beyzbol sopası", "baseball glove": "beyzbol eldiveni", "skateboard": "kaykay",
    "surfboard": "sörf tahtası", "tennis racket": "tenis raketi", "bottle": "şişe",
    "wine glass": "kadeh", "cup": "fincan", "fork": "çatal", "knife": "bıçak", "spoon": "kaşık",
    "bowl": "kase", "banana": "muz", "apple": "elma", "sandwich": "sandviç", "orange": "portakal",
    "broccoli": "brokoli", "carrot": "havuç", "hot dog": "sosisli", "pizza": "pizza",
    "donut": "donut", "cake": "pasta", "chair": "sandalye", "couch": "kanepe",
    "potted plant": "saksı bitkisi", "bed": "yatak", "dining table": "masa", "toilet": "klozet",
    "tv": "televizyon", "laptop": "dizüstü bilgisayar", "mouse": "fare", "remote": "kumanda",
    "keyboard": "klavye", "cell phone": "cep telefonu", "microwave": "mikrodalga", "oven": "fırın",
    "toaster": "tost makinesi", "sink": "lavabo", "refrigerator": "buzdolabı", "book": "kitap",
    "clock": "saat", "vase": "vazo", "scissors": "makas", "teddy bear": "oyuncak ayı",
    "hair drier": "saç kurutma makinesi", "toothbrush": "diş fırçası",
}


def kutulari_ciz(resim, sonuc, saydam=False):
    """YOLO sonucunu PIL ile çizer (Türkçe karakterler için). saydam=True → yalnızca kutular."""
    tuval = Image.new("RGBA", resim.size, (0, 0, 0, 0)) if saydam else resim.copy()
    ciz = ImageDraw.Draw(tuval)
    boy = max(14, min(resim.size) // 32)                       # yazı, görüntü boyutuyla ölçeklenir
    yazi = ImageFont.truetype(YAZI_DOSYASI, boy)
    k = sonuc.boxes
    kimlikler = k.id.int().tolist() if k.id is not None else [None] * len(k)
    for (x1, y1, x2, y2), sinif, p, kimlik in zip(k.xyxy.tolist(), k.cls.int().tolist(),
                                                   k.conf.tolist(), kimlikler):
        renk = RENKLER[(kimlik if kimlik is not None else sinif) % len(RENKLER)]
        ad = COCO_TR.get(sonuc.names[sinif], sonuc.names[sinif])
        etiket = (f"#{kimlik} " if kimlik is not None else "") + f"{ad} %{p * 100:.0f}"
        ciz.rectangle([x1, y1, x2, y2], outline=renk, width=max(3, boy // 6))
        ty = max(0, y1 - boy - 6)
        ciz.rectangle([x1, ty, x1 + ciz.textlength(etiket, font=yazi) + 8, ty + boy + 6], fill=renk)
        ciz.text((x1 + 4, ty + 2), etiket, fill="white", font=yazi)
    return tuval


def nesneleri_bul(resim, goster=True, baslik=""):
    """PIL görüntüsünde nesneleri bulur; kutulu görüntüyü ve sayımı gösterir."""
    sonuc = yolo(resim, verbose=False)[0]
    sayim = Counter(COCO_TR.get(sonuc.names[c], sonuc.names[c]) for c in sonuc.boxes.cls.int().tolist())
    if goster:
        plt.figure(figsize=(8, 8)); plt.imshow(kutulari_ciz(resim, sonuc)); plt.axis("off")
        plt.title(baslik or ", ".join(f"{n} {ad}" for ad, n in sayim.most_common()) or "Nesne bulunamadı")
        plt.show()
    return sonuc, sayim


ornek = Image.open(ASSETS / "bus.jpg").convert("RGB")      # ultralytics ile gelen örnek fotoğraf
sonuc, sayim = nesneleri_bul(ornek)

print(f"{'nesne':<16}{'güven':>8}   kutu (x1, y1, x2, y2)")
for c, p, kutu in zip(sonuc.boxes.cls.int().tolist(), sonuc.boxes.conf.tolist(), sonuc.boxes.xyxy.tolist()):
    print(f"{COCO_TR[sonuc.names[c]]:<16}{p:>8.2f}   {[round(v) for v in kutu]}")

### Kendi fotoğrafınızda nesne arayın

Bilgisayarınızdan bir fotoğraf seçin — sokak, masa üstü, mutfak, sınıf...
Hangi nesneleri tanıyor, hangilerini kaçırıyor?

In [ ]:
for kendi in resim_yukle():
    nesneleri_bul(kendi)

---
## 8 · Kamera — fotoğraf çek, model tanısın

Hücreyi çalıştırınca tarayıcı **kamera izni** ister; izin verin. Görüntü gelince
**📸 Fotoğraf çek** düğmesine basın. Çekilen kare hem YOLO'ya (nesneler) hem
DeepFace'e (yüzler) gider.

> Görüntü yalnızca tarayıcınız ile kendi Colab oturumunuz arasında gidip gelir,
> hiçbir yere kaydedilmez.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js

JS_FOTO = """
async function fotoCek(kalite) {
  const div = document.createElement('div');
  const dugme = document.createElement('button');
  dugme.textContent = '📸 Fotoğraf çek';
  dugme.style.cssText = 'font-size:16px;padding:6px 14px;margin:6px 0;background:#C08A2E;color:white;border:0;border-radius:6px;cursor:pointer';
  const video = document.createElement('video');
  video.style.display = 'block';
  video.setAttribute('playsinline', '');
  const akis = await navigator.mediaDevices.getUserMedia({video: true});
  div.appendChild(dugme); div.appendChild(video);
  document.body.appendChild(div);
  video.srcObject = akis;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise(r => dugme.onclick = r);
  const tuval = document.createElement('canvas');
  tuval.width = video.videoWidth; tuval.height = video.videoHeight;
  tuval.getContext('2d').drawImage(video, 0, 0);
  akis.getVideoTracks()[0].stop();
  div.remove();
  return tuval.toDataURL('image/jpeg', kalite);
}
"""


def dataurl_to_pil(veri):
    return Image.open(BytesIO(b64decode(veri.split(",", 1)[1]))).convert("RGB")


def pil_to_dataurl(resim, bicim="PNG"):
    tampon = BytesIO(); resim.save(tampon, format=bicim)
    return f"data:image/{bicim.lower()};base64," + b64encode(tampon.getvalue()).decode()


def kameradan_foto(kalite=0.9):
    display(Javascript(JS_FOTO))
    return dataurl_to_pil(eval_js(f"fotoCek({kalite})"))


try:
    kare = kameradan_foto()
except Exception as e:
    kare = None
    print("⚠ Kamera açılamadı — tarayıcıda kamera iznini verin ya da bölüm 5'teki yüklemeyi kullanın.\n ", e)

if kare is not None:
    kare.save("kamera.jpg")
    _, sayim = nesneleri_bul(kare, baslik="YOLO — kameradaki nesneler")
    print("Nesneler:", ", ".join(f"{n} {ad}" for ad, n in sayim.most_common()) or "yok")
    foto_analiz_et(kare, "DeepFace — kameradaki yüzler")

---
## 9 · Canlı kamera — nesne takibi (tracking)

Tespit her karede "burada bir kişi var" der. **Takip** ise bir adım ileri gider:
"bu, az önceki **#3 numaralı kişi**." YOLO her kareyi işler, **ByteTrack**
algoritması kutuları kareler arasında eşleştirip her nesneye kalıcı bir kimlik verir.

- Kamera açılınca önüne bir şeyler getirin: telefon, bardak, kitap, el...
- Kutuların üzerindeki `#numara` takip kimliği; nesne kadrajda kaldıkça aynı kalır
- **Durdurmak için görüntüye tıklayın** (ya da süre dolunca kendiliğinden durur)

> CPU'da saniyede birkaç kare işlenir; akıcı değil ama canlı. Bu defter CPU
> çalışma zamanında test edildi — çalışma zamanı türünü değiştirmeyin.

> **Konuşma notu:** Mağazalardaki müşteri sayımı, trafikteki araç takibi, sporda
> oyuncu analizi — hepsi bu iki adımın birleşimi: **tespit + eşleştirme**.

In [ ]:
JS_AKIS = """
var video, div = null, akis, yakalama, katmanImg, etiketDiv;
var bekleyen = null, kapatIstegi = false;
const BOS = 'data:image/gif;base64,R0lGODlhAQABAAAAACH5BAEKAAEALAAAAAABAAEAAAICTAEAOw==';

function akisKapat() {
  if (akis) akis.getVideoTracks()[0].stop();
  if (div) div.remove();
  video = null; div = null; akis = null; katmanImg = null; etiketDiv = null; yakalama = null;
}

function animasyon() {
  if (!kapatIstegi) window.requestAnimationFrame(animasyon);
  if (bekleyen) {
    let sonuc = '';
    if (!kapatIstegi) {
      yakalama.getContext('2d').drawImage(video, 0, 0, 640, 480);
      sonuc = yakalama.toDataURL('image/jpeg', 0.8);
    }
    const b = bekleyen; bekleyen = null; b(sonuc);
  }
}

async function akisBaslat() {
  if (div !== null) return;
  div = document.createElement('div');
  div.style.cssText = 'border:3px solid #C08A2E;border-radius:8px;padding:6px;width:640px;font-family:sans-serif';
  etiketDiv = document.createElement('div');
  etiketDiv.style.cssText = 'font-weight:bold;padding:4px;color:#15263F';
  const kutu = document.createElement('div');
  kutu.style.cssText = 'position:relative;width:640px;height:480px';
  video = document.createElement('video');
  video.width = 640; video.height = 480; video.setAttribute('playsinline', '');
  katmanImg = document.createElement('img');
  katmanImg.src = BOS; katmanImg.width = 640; katmanImg.height = 480;
  katmanImg.style.cssText = 'position:absolute;left:0;top:0;cursor:pointer';
  katmanImg.onclick = () => { kapatIstegi = true; };
  const not = document.createElement('div');
  not.textContent = 'Durdurmak için görüntüye tıklayın.';
  not.style.cssText = 'font-size:12px;color:#666;padding:4px';
  kutu.appendChild(video); kutu.appendChild(katmanImg);
  div.appendChild(etiketDiv); div.appendChild(kutu); div.appendChild(not);
  document.body.appendChild(div);
  akis = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  video.srcObject = akis; await video.play();
  yakalama = document.createElement('canvas'); yakalama.width = 640; yakalama.height = 480;
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  window.requestAnimationFrame(animasyon);
}

async function kareAl(etiket, katman) {
  if (kapatIstegi) { akisKapat(); kapatIstegi = false; return ''; }
  await akisBaslat();
  katmanImg.src = katman || BOS;
  etiketDiv.textContent = etiket;
  return await new Promise(r => bekleyen = r);
}
"""


def canli_takip(sure_sn=90, imgsz=480):
    """Kamerayı açar; her kareyi YOLO + ByteTrack ile işler, kutuları canlı çizer."""
    takipci = YOLO(YOLO_AGIRLIK)          # her oturumda temiz takip hafızası
    display(Javascript(JS_AKIS))
    etiket, katman, kare_sayisi, baslangic = "Kamera açılıyor...", "", 0, time.time()
    gorulen = set()
    while time.time() - baslangic < sure_sn:
        try:
            veri = eval_js(f"kareAl({json.dumps(etiket)}, {json.dumps(katman)})")
        except Exception as e:
            print("⚠ Kamera açılamadı — tarayıcıda kamera iznini verin ya da aşağıdaki video yüklemeyi kullanın.\n ", e)
            break
        if not veri:
            break
        kare = dataurl_to_pil(veri)
        sonuc = takipci.track(kare, persist=True, imgsz=imgsz, verbose=False)[0]
        katman = pil_to_dataurl(kutulari_ciz(kare, sonuc, saydam=True))
        if sonuc.boxes.id is not None:
            gorulen.update(zip(sonuc.boxes.id.int().tolist(), sonuc.boxes.cls.int().tolist()))
        kare_sayisi += 1
        sayim = Counter(COCO_TR.get(sonuc.names[c], sonuc.names[c]) for c in sonuc.boxes.cls.int().tolist())
        fps = kare_sayisi / (time.time() - baslangic)
        etiket = f"{fps:.1f} kare/sn · " + (", ".join(f"{n} {ad}" for ad, n in sayim.most_common()) or "nesne yok")
    display(Javascript("if (typeof akisKapat === 'function') akisKapat();"))
    print(f"Takip bitti: {kare_sayisi} kare işlendi, {len(gorulen)} farklı nesneye kimlik verildi.")
    ozet = Counter(COCO_TR.get(yolo.names[c], yolo.names[c]) for _, c in gorulen)
    if ozet:
        print("Kimlik verilenler:", ", ".join(f"{n} {ad}" for ad, n in ozet.most_common()))


canli_takip(sure_sn=90)

### Kamera yoksa: bilgisayardan video yükleyin

Kısa bir video seçin (telefonla çekilmiş 5–15 saniyelik bir sokak, koridor ya da
masa görüntüsü yeterli). İlk ~10 saniyesi işlenir, kimlikli kutularla yeni bir
video olarak aşağıda oynatılır.

In [ ]:
import glob, shutil, subprocess
from IPython.display import HTML, display


def videoda_takip(dosya, en_fazla_sn=10, kare_hizi=10, imgsz=480):
    """Videonun ilk saniyelerini kareye böler (ffmpeg), her kareyi takip eder, kutulu videoyu oynatır."""
    for klasor in ["_kareler", "_cizili"]:
        shutil.rmtree(klasor, ignore_errors=True); os.makedirs(klasor)
    subprocess.run(["ffmpeg", "-loglevel", "error", "-i", dosya, "-t", str(en_fazla_sn),
                    "-vf", f"fps={kare_hizi},scale='min(960,iw)':-2", "_kareler/%04d.jpg"], check=True)
    kareler = sorted(glob.glob("_kareler/*.jpg"))
    if not kareler:
        print("⚠ Video okunamadı."); return
    takipci, gorulen = YOLO(YOLO_AGIRLIK), set()
    for k in kareler:
        kare = Image.open(k).convert("RGB")
        sonuc = takipci.track(kare, persist=True, imgsz=imgsz, verbose=False)[0]
        if sonuc.boxes.id is not None:
            gorulen.update(sonuc.boxes.id.int().tolist())
        kutulari_ciz(kare, sonuc).save(k.replace("_kareler", "_cizili"))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(kare_hizi),
                    "-i", "_cizili/%04d.jpg", "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                    "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "takip.mp4"], check=True)
    print(f"{len(kareler)} kare işlendi, {len(gorulen)} farklı nesneye kimlik verildi.")
    veri = b64encode(open("takip.mp4", "rb").read()).decode()
    display(HTML(f'<video width=640 controls autoplay loop muted src="data:video/mp4;base64,{veri}"></video>'))


from google.colab import files
for ad, icerik in files.upload().items():
    open(ad, "wb").write(icerik)
    videoda_takip(ad)

---
## 10 · Sınırlar ve sorumluluk

Bu demoyu kapatmadan önce söylenmesi gereken birkaç şey var:

- **Yaş tahmini yaklaşıktır.** Model, eğitildiği veri setindeki insanlara benziyorsanız
  iyi çalışır; benzemiyorsanız ciddi şekilde şaşar.
- **Cinsiyet ikili değildir.** Model iki sınıfa zorlanmış durumda; bu bir gerçeklik
  değil, veri setinin kısıtı.
- **Duygu, yüz ifadesinden okunmaz.** Model yüz kaslarının konumunu sınıflandırıyor;
  insanın ne hissettiğini değil. Gülümseyen biri mutlu olmayabilir.
- **Yanlılık gerçektir.** Yüz veri setleri tarihsel olarak belirli gruplara ağırlık
  verdi; doğruluk gruplara göre belirgin şekilde değişiyor.
- **Takip, gözetime çok yakındır.** Kişileri kareler boyunca izlemek teknik olarak
  kolay; kimin, kimi, hangi amaçla izlediği sorusu teknik değil.
- **Biyometrik veri hassas veridir.** KVKK ve GDPR kapsamında özel nitelikli kişisel
  veri sayılır; rıza olmadan işlenemez. Bu defterde fotoğraf ve kamera görüntüsü
  Colab oturumunda kalıyor, hiçbir yere gitmiyor.
- **Bu modeller işe alım, güvenlik, kredi gibi kararlarda kullanılmamalı.**

> **Konuşma notu:** Bu bölümü atlamayın. Bir modelin çalışıyor olması, o modeli
> kullanmanın doğru olduğu anlamına gelmiyor. Teknik yeterlilik ile etik yeterlilik
> ayrı şeyler.

---
## Özet

Bugün anlattığımız her parçayı çalışırken gördük:

| Adım | Arkasında ne var |
|---|---|
| Yüzü bulmak | RetinaFace — CNN, filtreler görüntüde gezdi |
| Kırpıp ölçeklemek | Normalizasyon — 150.528 sayı, 0–1 aralığında |
| Duygu | 7 sınıflı softmax, toplam %100 |
| Yaş | Regresyon — tek sayı |
| Cinsiyet | İkili sınıflandırma |
| Kimlik | 512 boyutlu embedding, kosinüs benzerliği |
| Nesneler | YOLO — tek bakışta konum (regresyon) + sınıf (olasılık) |
| Takip | Tespit + kareler arası eşleştirme = kalıcı kimlik |

Ve hepsinin ortak noktası: **hiçbir kuralı biz yazmadık.**
Bütün bu davranışlar veriden öğrenildi.

---

### Denemek isterseniz

- Aynı kişinin gülümseyen ve nötr iki fotoğrafını karşılaştırın — duygu değişiyor mu?
- Yan dönmüş, gözlüklü, yarısı kapalı bir yüz verin — model nerede zorlanıyor?
- Kalabalık bir fotoğraf verin — kaç yüz, kaç nesne buluyor?
- `YOLO_AGIRLIK = "yolo26s.pt"` deneyin: daha büyük model, daha isabetli, daha yavaş
- Canlı takipte bir nesneyi kadrajdan çıkarıp geri getirin — aynı kimliği alıyor mu?

*Deep Learning Master Class · Kuantum Bootcamp · Veysel Murat Görken*

*Kaynak: [generative-ai-workshop](https://github.com/grknc/generative-ai-workshop)*